# CPBL 主客場勝率預測 — Colab 一鍵 pipeline

**用法：** `Runtime → Run all`（或由上往下逐格跑）。可重複執行 —— Cell 2 會自動更新程式碼且**不會**刪掉已下載的資料。

| Cell | 做什麼 |
|---|---|
| 2 | clone / 更新 repo（強制對齊遠端，殺 stale） |
| 3 | 裝套件（lightgbm / shap / xgboost） |
| 4 | 抓 rebas **2024+2023**（`USE_2023` 預設 `True`） |
| 4b | **投手資料診斷**（貼整段輸出回 Claude） |
| 5 | 跑 step1→step1b(天氣)→step2→step3 |
| 6 | 印 `_final_metrics.json` + 三張圖 |
| 8 | (選) 把產出推回 GitHub |

跑完把 **Cell 4b 整段 + Cell 5 結尾 + Cell 6 的 JSON** 貼回給 Claude。

In [ ]:
# === Cell 2 — clone / 更新 repo（idempotent；reset --hard 不動 gitignored 的 data/raw）===
import os, subprocess
REPO = "/content/data_science_final_project"
URL  = "https://github.com/jiangjiangian/data_science_final_project.git"
BR   = "claude/setup-main-agent-BhYTE"
if not os.path.isdir(os.path.join(REPO, ".git")):
    subprocess.run(["git","clone","--branch",BR,"--single-branch",URL,REPO], check=True)
os.chdir(REPO)
subprocess.run(["git","fetch","origin"], check=True)
subprocess.run(["git","reset","--hard",f"origin/{BR}"], check=True)
print(subprocess.run(["git","log","-1","--oneline"],capture_output=True,text=True).stdout.strip())
print("scripts/:", sorted(os.listdir("scripts")))
print("→ 最後一行 commit 必須 >= 3d808dc，scripts/ 只有 .py")

In [ ]:
# === Cell 3 — 套件 ===
import subprocess, sys
subprocess.run([sys.executable,"-m","pip","install","-q","lightgbm","shap","xgboost"], check=True)
import sklearn, xgboost, lightgbm, shap
print("sklearn", sklearn.__version__, "| xgb", xgboost.__version__,
      "| lgb", lightgbm.__version__, "| shap", shap.__version__)

In [ ]:
# === Cell 4 — 抓 rebas 資料（每個新 runtime 必跑；zip 不在 git）===
import os, subprocess, glob
USE_2023 = True   # 預設 True：單 2024 已實證無樣本外訊號；2023 把 N 366→~700

os.makedirs("data/raw", exist_ok=True)
B = "https://github.com/rebas-tw/rebas.tw-open-data/releases/download"
urls = [
    f"{B}/v0.1.0-2024/CPBL-2024-OpenData.zip",
    f"{B}/v0.1.0-2024/CPBL-2024-Challenge-OpenData.zip",
    f"{B}/v0.1.0-2024/CPBL-2024-TaiwanSeries-OpenData.zip",
]
if USE_2023:
    urls += [
        f"{B}/v0.1.0-2023.0/CPBL-2023-G1-G150-OpenData.zip",
        f"{B}/v0.1.0-2023.1/CPBL-2023-G151-G300-OpenData.zip",
        f"{B}/v0.1.0-2023.1/CPBL-2023-Challenge-OpenData.zip",
        f"{B}/v0.1.0-2023.1/CPBL-2023-TaiwanSeries-OpenData.zip",
    ]
for u in urls:
    stem = u.split("/")[-1][:-4]                      # 去掉 .zip
    fn = f"data/raw/{stem}.zip"
    subprocess.run(["wget","-q","-O",fn,u], check=True)
    subprocess.run(["unzip","-o","-q",fn,"-d",f"data/raw/{stem}"], check=True)
# 2024=ASCII (CPBL-2024-OpenData.json) / 2023=中文 (中職2023年-OpenData.json
# 等4種)；只配共同 token 'OpenData'，自動排除 per-game *-G<N>.json
js = sorted(glob.glob("data/raw/**/*OpenData*.json", recursive=True))
print(len(js), "combined JSON:")
for j in js: print("  ", j)
need = 7 if USE_2023 else 3
assert len(js) >= need, f"❌ 預期 >= {need} 個合併檔，只有 {len(js)} — 看上面清單"

In [ ]:
# === Cell 4b — 投手資料診斷（貼【整段】輸出回 Claude → gating 投手特徵程式碼）===
import json, glob, collections, statistics
files = sorted(glob.glob("data/raw/**/*OpenData*.json", recursive=True))
games = []
for f in files: games += json.load(open(f))
print("files:", [f.split("/")[-1] for f in files], "| total games:", len(games))
g = games[0]
print("game keys:", list(g.keys()))
pb = g.get("homePitcherBox") or []
print("homePitcherBox rows:", len(pb))
print("ONE pitcher row:", json.dumps(pb[0], ensure_ascii=False) if pb else "NONE")
print("home order values:", [r.get("order") for r in pb])
bad=0; starts=collections.Counter(); team_sp=collections.defaultdict(set)
seasons=collections.Counter()
for gg in games:
    seasons[str(gg.get("seasonId") or gg.get("season"))[:4]] += 1
    for side,tk in (("homePitcherBox","homeTeam"),("awayPitcherBox","awayTeam")):
        rows=gg.get(side) or []
        s=[r for r in rows if r.get("order")==1]
        if len(s)!=1: bad+=1
        elif s:
            pid=s[0].get("playerId"); starts[pid]+=1; team_sp[gg.get(tk)].add(pid)
sv=list(starts.values())
print("games by season:", dict(seasons))
print("game-sides WITHOUT exactly one order==1:", bad, "/", 2*len(games))
print("distinct starters:", len(sv),
      "| starts/starter mean/median/max:",
      round(statistics.mean(sv),1) if sv else 0,
      statistics.median(sv) if sv else 0, max(sv) if sv else 0)
print("starters per team:", {t:len(s) for t,s in team_sp.items()})

In [ ]:
# === Cell 5 — 全 pipeline（step1 → step1b 天氣 → step2 → step3）===
# step1b 缺座標 / 天氣缺漏 >50% 會直接 SystemExit（不靜默塞 NA）
!python3 scripts/run_all.py

In [ ]:
# === Cell 6 — 結果 + 圖 ===
import os, json, pathlib
from IPython.display import Image, display
p = pathlib.Path("Results/eval/_final_metrics.json")
if p.exists():
    print(json.dumps(json.loads(p.read_text()), indent=2, ensure_ascii=False))
else:
    print("❌ _final_metrics.json 沒生成 — 看 Cell 5 紅字並貼回給 Claude")
for f in ["model_comparison.png","calibration.png","shap_summary.png"]:
    fp = f"Results/figures/{f}"
    if os.path.exists(fp):
        print("\n"+fp); display(Image(fp))
pc = pathlib.Path("Results/eval/predictions.csv")
print("\npredictions.csv:", "OK" if pc.exists() else "MISSING")

### Cell 8（選用）把產出推回 GitHub

只有要保存 `Results/eval/*` 給之後 Shiny 用才需要。需要一組有 repo 權限的 GitHub PAT。

In [ ]:
# === Cell 8 (選用) — 推回產出 ===
from getpass import getpass
import subprocess
tok = getpass("GitHub PAT (repo scope): ")
subprocess.run(["git","config","user.email","colab@run.local"], check=True)
subprocess.run(["git","config","user.name","colab"], check=True)
subprocess.run(["git","add","-f","Results/eval","Results/figures"], check=True)
subprocess.run(["git","commit","-m","data: Run B artifacts (Colab)"], check=False)
url = f"https://{tok}@github.com/jiangjiangian/data_science_final_project.git"
r = subprocess.run(["git","push",url,"HEAD:claude/setup-main-agent-BhYTE"], capture_output=True, text=True)
print(r.stdout or r.stderr or "pushed ✓")